# Design and implement data modeling with Azure Databricks

## Industry context: Financial Services

This demo uses a **fictional financial services company** called **FinServ Demo** — an investment bank that manages client portfolios, executes trades across asset classes (equities, fixed income, ETFs, and FX), and maintains real-time market data.

The data model covers:
- **Clients** — 400 clients across Americas, EMEA, and APAC, each with KYC status, risk rating, and segment
- **Accounts** — Current, savings, investment, pension, and trading accounts per client
- **Trades** — 8,000+ equity, bond, ETF, and FX trades over 2 years
- **Market data** — Daily OHLCV price data for 26 securities across a full year

This context makes data modeling decisions tangible: partitioning by trade date for regulatory queries, SCD Type 2 for client risk rating changes, change data feed for audit trails, and liquid clustering for multi-dimensional trade analysis.


In [ ]:
from scripts.setup import setup
setup(spark)

## Design ingestion logic and data source configuration

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-data-ingestion-logic.png)

### Discussion: Extraction patterns for trade data

FinServ Demo ingests trade data from its order management system (OMS). Trades arrive in different ways depending on the source:

| Pattern | When to use | Finance example |
|---|---|---|
| **Full extraction** | Small reference tables, no change tracking | Market reference data (currency codes, exchange codes) |
| **Incremental extraction** | Large tables with a reliable timestamp column | Trades and settlements — `trade_date` is the watermark |
| **Streaming extraction** | Real-time event processing, tick data | Market price feeds, intraday order book updates |


## Choose a data ingestion tool

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-data-ingestion-tool.png)

### Lakeflow Connect

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-lakeflow-connect.png)

### Auto Loader

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-auto-loader.png)

### COPY INTO

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-copy-into.png)

### Azure Data Factory

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-azure-data-factory.png)

### Apply a decision framework

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/3-decision-framework.png)

## Choose a data table format

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-data-table-format.png)

### How Parquet stores data differently

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/4-parquet-file-layout.png)

## Design and implement a data partitioning scheme

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-data-partitioning-scheme.png)

### 🎯 DEMO: Partitioning the trades table by trade date

Compliance teams regularly query FinServ Demo trades by date range — "show all trades between 2023-Q1 and 2023-Q2". Without partitioning, every query scans all 8,000 trade records. With daily partitioning by `trade_date`, the engine reads only the relevant date folders.

> **When to partition**: The trades table has 8,000 rows in this demo — much less than the 1 TB Databricks threshold. In production at FinServ Demo (millions of trades over years), partitioning by `trade_date` becomes essential.

> **Liquid clustering** is the modern recommendation — we'll see that in the clustering demo.


In [ ]:
%sql
-- Create a partitioned version of the trades table.
-- PARTITIONED BY (trade_date) puts each day's trades in a separate directory.
-- The query engine uses partition pruning to skip directories that don't match the filter.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE trades_partitioned
USING DELTA
PARTITIONED BY (trade_date)
AS
SELECT * FROM trades;

-- Show partition statistics
DESCRIBE EXTENDED trades_partitioned;


In [ ]:
%sql
-- Query with a date filter — the engine prunes all partition folders outside this range.
-- EXPLAIN shows the physical plan and confirms partition pruning is applied.

EXPLAIN
SELECT ticker, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_partitioned
WHERE trade_date BETWEEN '2023-01-01' AND '2023-03-31'
GROUP BY ticker
ORDER BY total_value DESC;


In [ ]:
%sql
-- Generated columns: derive the partition column from a timestamp.
-- In many OMS exports, trades include a precise timestamp. Using a generated
-- column, you can partition by date while keeping the full timestamp.

CREATE OR REPLACE TABLE trades_with_timestamp
(
    trade_id        STRING NOT NULL,
    account_id      STRING,
    ticker          STRING,
    asset_class     STRING,
    trade_type      STRING,
    quantity        INT,
    price           DOUBLE,
    total_value     DOUBLE,
    trade_timestamp TIMESTAMP,
    trade_date      DATE GENERATED ALWAYS AS (CAST(trade_timestamp AS DATE)),
    settlement_date DATE,
    status          STRING,
    region          STRING
)
USING DELTA
PARTITIONED BY (trade_date);

-- trade_date is auto-populated from trade_timestamp — no redundant column to maintain
DESCRIBE trades_with_timestamp;


## Choose a slowly changing dimension (SCD) type

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-scd-type.png)

### Compare SCD types

The following table summarizes when to select each SCD type:

| Consideration               | Type 0                        | Type 1                             | Type 2                                | Type 3                   |
| --------------------------- | ----------------------------- | ---------------------------------- | ------------------------------------- | ------------------------ |
| **History preserved**       | Original only                 | None                               | Complete                              | Limited                  |
| **Storage impact**          | Minimal                       | Minimal                            | High                                  | Moderate                 |
| **Query complexity**        | Simple                        | Simple                             | Complex                               | Moderate                 |
| **Common use cases**        | Constants, audit requirements | Error corrections, contact updates | Sales attribution, territory tracking | Before/after comparisons |
| **Fact table relationship** | Natural key                   | Natural key                        | Surrogate key                         | Natural key              |

## Implement a slowly changing dimension (SCD) type 2

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.implement-slowly-changing-dimension-type-2.png)

### 🎯 DEMO: Implementing SCD Type 2 for FinServ Demo clients

Client records at FinServ Demo change over time in two important ways:
- **Region** — a client is reassigned from EMEA to Americas because they relocate
- **Risk rating** — a credit review changes a client from Low to High risk

For compliance and portfolio attribution, we must know what a client's risk rating was **at the time of each trade**. SCD Type 2 preserves this history by inserting a new version row rather than overwriting the existing record.

The `client_staging` table simulates the latest data from the CRM system (upstream source). It contains:
- **60 updated records** (changed region or risk_rating for existing clients)
- **10 new clients** not yet in the dimension table


In [ ]:
%sql
-- Step 1: Create the client dimension table with SCD Type 2 tracking columns.
-- Surrogate key (client_sk) uniquely identifies each version.
-- Natural key (client_id) links versions to the same real-world client.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE client_dim (
    client_sk            BIGINT GENERATED ALWAYS AS IDENTITY,  -- surrogate key
    client_id            STRING NOT NULL,                        -- natural/business key
    first_name           STRING,
    last_name            STRING,
    email                STRING,
    country              STRING,
    region               STRING,
    segment              STRING,
    kyc_status           STRING,
    risk_rating          STRING,
    relationship_manager STRING,
    onboarding_date      DATE,
    valid_from           TIMESTAMP NOT NULL,
    valid_to             TIMESTAMP NOT NULL,
    is_current           BOOLEAN
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Initial load: treat all clients as current versions
INSERT INTO client_dim (
    client_id,
    first_name,
    last_name,
    email,
    country,
    region,
    segment,
    kyc_status,
    risk_rating,
    relationship_manager,
    onboarding_date,
    valid_from,
    valid_to,
    is_current
)
SELECT
    client_id, first_name, last_name, email, country,
    region, segment, kyc_status, risk_rating, relationship_manager,
    onboarding_date,
    CAST('2020-01-01' AS TIMESTAMP)    AS valid_from,
    CAST('9999-12-31' AS TIMESTAMP)    AS valid_to,
    true                               AS is_current
FROM trainer_demo.demo_06.clients;

SELECT COUNT(*) AS initial_rows FROM client_dim;


In [ ]:
%sql
-- Step 2: Apply changes from the staging table using a two-step MERGE pattern.
-- Step A — close the current row for records that changed (set valid_to and is_current = false)

MERGE INTO trainer_demo.demo_06.client_dim AS target
USING (
    SELECT
        s.client_id,
        s.region,
        s.risk_rating,
        s.kyc_status,
        s.relationship_manager,
        current_timestamp() AS effective_ts
    FROM trainer_demo.demo_06.client_staging AS s
) AS updates
ON target.client_id = updates.client_id
   AND target.is_current = true
WHEN MATCHED AND (
    target.region               <> updates.region OR
    target.risk_rating          <> updates.risk_rating OR
    target.kyc_status           <> updates.kyc_status OR
    target.relationship_manager <> updates.relationship_manager
) THEN UPDATE SET
    target.valid_to    = updates.effective_ts,
    target.is_current  = false;

SELECT COUNT(*) AS expired_rows FROM trainer_demo.demo_06.client_dim WHERE is_current = false;


In [ ]:
%sql
-- Step B: Insert new current versions for records that were just closed,
-- and also insert brand-new clients that don't yet exist in the dimension.

INSERT INTO trainer_demo.demo_06.client_dim (
    client_id,
    first_name,
    last_name,
    email,
    country,
    region,
    segment,
    kyc_status,
    risk_rating,
    relationship_manager,
    onboarding_date,
    valid_from,
    valid_to,
    is_current
)
SELECT
    s.client_id, s.first_name, s.last_name, s.email,
    s.country, s.region, s.segment, s.kyc_status,
    s.risk_rating, s.relationship_manager, s.onboarding_date,
    current_timestamp()            AS valid_from,
    CAST('9999-12-31' AS TIMESTAMP) AS valid_to,
    true                            AS is_current
FROM trainer_demo.demo_06.client_staging AS s
LEFT JOIN trainer_demo.demo_06.client_dim AS d
    ON s.client_id = d.client_id AND d.is_current = true
WHERE d.client_id IS NULL   -- new client (not in dim yet)
   OR (                     -- existing client whose current row was just closed
       d.client_id IS NOT NULL
       AND d.is_current = false
   );

SELECT
    is_current,
    COUNT(*) AS rows
FROM trainer_demo.demo_06.client_dim
GROUP BY is_current
ORDER BY is_current;


In [ ]:
%sql
-- Step C: Query the history of a specific client to see risk rating changes over time.
-- This is the SCD Type 2 "complete history" query pattern.

SELECT
    client_id,
    first_name || ' ' || last_name AS client_name,
    risk_rating,
    region,
    relationship_manager,
    valid_from,
    valid_to,
    is_current
FROM trainer_demo.demo_06.client_dim
WHERE client_id = 'CLT-00001'
ORDER BY valid_from;


In [ ]:
%sql
-- Point-in-time query: what was a client's risk rating at the time of a specific trade?
-- Join the fact table to the dimension using the surrogate key's validity window.
-- This is the key query pattern enabled by SCD Type 2.

SELECT
    t.trade_id,
    t.trade_date,
    t.ticker,
    t.total_value,
    c.risk_rating    AS risk_at_trade_time,
    c.region         AS region_at_trade_time,
    c.relationship_manager
FROM trainer_demo.demo_06.trades AS t
JOIN trainer_demo.demo_06.accounts AS a
    ON t.account_id = a.account_id
JOIN trainer_demo.demo_06.client_dim AS c
    ON a.client_id = c.client_id
    AND CAST(t.trade_date AS TIMESTAMP) >= c.valid_from
    AND CAST(t.trade_date AS TIMESTAMP) <  c.valid_to
ORDER BY t.trade_date DESC
LIMIT 10;


## Design and implement a temporal (history) table to record changes over time

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-temporal-history-table.png)

### 🎯 DEMO: Temporal history table for trade audit trail

Regulators require FinServ Demo to maintain a complete audit trail of every change to trade records. If a compliance officer asks "what was the status of trade TRD-0000001 on January 15?", the team needs to answer precisely — not just return the current value.

Delta Lake's **change data feed (CDF)** captures every row-level change (insert, update, delete) with metadata. Combined with a persistent history table populated via Structured Streaming, this creates an immutable audit trail.


In [ ]:
%sql
-- Step 1: Enable change data feed on the trades table.
-- CDF records every insert, update, and delete with commit timestamps.
-- IMPORTANT: CDF only records changes from this point forward.

USE trainer_demo.demo_06;

ALTER TABLE trades
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Confirm CDF is enabled
SHOW TBLPROPERTIES trades;


In [ ]:
%sql
-- Step 2: Make some changes to the trades table to generate CDF events.
-- Simulate a compliance correction: update failed trades and delete a cancelled one.

UPDATE trainer_demo.demo_06.trades
SET status = 'Settled'
WHERE status = 'Failed'
  AND trade_date < '2023-06-01';

DELETE FROM trainer_demo.demo_06.trades
WHERE trade_id = 'TRD-0000050' AND status = 'Pending';

SELECT COUNT(*) AS total_trades FROM trainer_demo.demo_06.trades;


In [ ]:
%sql
-- Step 3: Query the change data feed to see what changed.
-- _change_type values:
--   'insert'           — new row was inserted
--   'update_preimage'  — values BEFORE the update
--   'update_postimage' — values AFTER the update
--   'delete'           — row was deleted

SELECT
    trade_id,
    status,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('trainer_demo.demo_06.trades', 1)
ORDER BY _commit_timestamp, trade_id
LIMIT 30;


In [ ]:
%sql
-- Step 4: Time travel — query the trades table as it was before the corrections.
-- This is useful for "as-of" regulatory reports and reconciliation.

-- View the full transaction history first
DESCRIBE HISTORY trainer_demo.demo_06.trades;


In [ ]:
%sql
-- Query the trades table at version 0 (before any corrections).
-- Version 0 = the initial data load. Use this to compare current vs original.

SELECT status, COUNT(*) AS count
FROM trainer_demo.demo_06.trades VERSION AS OF 0
GROUP BY status
ORDER BY count DESC;


In [ ]:
# Step 5: Persist the change history to a long-term audit table.
# Delta Lake's default retention is 7 days — not enough for financial compliance.
# Use Structured Streaming with readChangeFeed to build a permanent audit trail.

# Create destination audit table
spark.sql("""
    CREATE TABLE IF NOT EXISTS trainer_demo.demo_06.trades_audit_history
    USING DELTA
    AS SELECT
        trade_id, account_id, ticker, trade_type, total_value, trade_date, status,
        CAST(NULL AS STRING) AS _change_type,
        CAST(NULL AS BIGINT) AS _commit_version,
        CAST(NULL AS TIMESTAMP) AS _commit_timestamp
    FROM trainer_demo.demo_06.trades
    WHERE 1=0
""")

# Stream changes from trade table into audit history (trigger once = process all pending changes)
audit_query = (
    spark.readStream
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table("trainer_demo.demo_06.trades")
    .writeStream
        .option("checkpointLocation", "/tmp/checkpoints/trades_audit")
        .trigger(availableNow=True)
        .toTable("trainer_demo.demo_06.trades_audit_history")
)

audit_query.awaitTermination()
print(f"Audit rows captured: {spark.table('trainer_demo.demo_06.trades_audit_history').count():,}")


In [ ]:
%sql
-- Step 6: Configure extended retention for compliance requirements.
-- Financial regulators often require 7+ years of audit data. This setting
-- ensures Delta time travel stays available longer than the default 7 days.

ALTER TABLE trainer_demo.demo_06.trades
SET TBLPROPERTIES (
    delta.logRetentionDuration         = 'interval 90 days',
    delta.deletedFileRetentionDuration = 'interval 90 days'
);

SHOW TBLPROPERTIES trainer_demo.demo_06.trades;


## Choose granularity on a column or table based on requirements

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/9-understand-what-granularity-represents.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-granularity.png)

### 🎯 DEMO: Choosing granularity for market data and trade fact tables

FinServ Demo stores market price data at different granularities depending on the use case:

| Use case | Needed granularity | Table |
|---|---|---|
| Daily P&L reports | Daily OHLCV per ticker | `market_data` (already daily) |
| Performance attribution | Monthly average close per asset class | Pre-aggregated summary table |
| Tick data analysis | Per-second price changes | Not captured (would require streaming) |

**Key insight**: The setup creates `market_data` at daily granularity. We can always aggregate up to monthly, but we cannot disaggregate down to intraday. Always capture the finest granularity your budget allows.


In [ ]:
%sql
-- View daily granularity: each row = one ticker for one trading day
SELECT
    price_date,
    ticker,
    open_price,
    high_price,
    low_price,
    close_price,
    volume,
    asset_class
FROM trainer_demo.demo_06.market_data
WHERE ticker = 'AAPL'
ORDER BY price_date
LIMIT 10;


In [ ]:
%sql
-- Aggregate to monthly granularity for executive performance dashboards.
-- Once aggregated, intraday volatility information is lost — we only see
-- monthly open, high, low, close, and total volume.

SELECT
    DATE_TRUNC('MONTH', price_date)     AS month,
    ticker,
    FIRST_VALUE(open_price)
        OVER (PARTITION BY ticker, DATE_TRUNC('MONTH', price_date)
              ORDER BY price_date)       AS monthly_open,
    MAX(high_price)                      AS monthly_high,
    MIN(low_price)                       AS monthly_low,
    LAST_VALUE(close_price)
        OVER (PARTITION BY ticker, DATE_TRUNC('MONTH', price_date)
              ORDER BY price_date
              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS monthly_close,
    SUM(volume)                          AS monthly_volume
FROM trainer_demo.demo_06.market_data
WHERE ticker IN ('AAPL', 'MSFT', 'JPM')
GROUP BY month, ticker, price_date, open_price, close_price, high_price, low_price, volume
ORDER BY ticker, month
LIMIT 15;


In [ ]:
%sql
-- Create a materialized view for the monthly aggregate.
-- Azure Databricks automatically refreshes this as the underlying market_data changes.
-- The grain: one row per ticker per month.

USE trainer_demo.demo_06;

CREATE OR REPLACE MATERIALIZED VIEW market_data_monthly
AS
SELECT
    DATE_TRUNC('MONTH', price_date)  AS price_month,
    ticker,
    asset_class,
    AVG(close_price)                 AS avg_close,
    MAX(high_price)                  AS period_high,
    MIN(low_price)                   AS period_low,
    SUM(volume)                      AS total_volume,
    COUNT(*)                         AS trading_days
FROM trainer_demo.demo_06.market_data
GROUP BY DATE_TRUNC('MONTH', price_date), ticker, asset_class;

SELECT * FROM market_data_monthly WHERE ticker = 'AAPL' ORDER BY price_month LIMIT 12;


In [ ]:
%sql
-- Compare row counts at different granularity levels to illustrate storage impact.
-- Finer granularity = more rows = higher storage and compute cost.

SELECT 'Daily (atomic grain)' AS grain_level, COUNT(*) AS rows
FROM trainer_demo.demo_06.market_data

UNION ALL

SELECT 'Monthly aggregate' AS grain_level, COUNT(*) AS rows
FROM trainer_demo.demo_06.market_data_monthly

ORDER BY rows DESC;


## Choose managed vs unmanaged tables

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.choose-managed-vs-unmanaged-tables.png)

### 🎯 DEMO: Managed vs external tables in financial services

At FinServ Demo:
- **Managed tables** are used for all core data that lives exclusively in the Databricks lakehouse — trades, client data, market prices. Unity Catalog controls the data lifecycle including automatic cleanup.
- **External tables** are used when the risk management team's on-premises system needs read access to the same underlying Parquet files, or when migrating legacy Hive-managed tables without moving data.

> **Key difference to show**: Drop a managed table → data is scheduled for removal. Drop an external table → data files remain intact.


In [ ]:
%sql
-- Managed table: Unity Catalog owns both metadata AND data files.
-- When dropped, data files are automatically removed after 8 days (UNDROP is possible within that window).

USE trainer_demo.demo_06;

CREATE TABLE IF NOT EXISTS portfolio_positions_managed
USING DELTA
AS
SELECT
    a.account_id,
    t.ticker,
    t.asset_class,
    SUM(CASE WHEN t.trade_type = 'Buy' THEN t.quantity
             WHEN t.trade_type = 'Sell' THEN -t.quantity
             ELSE 0 END)   AS net_quantity,
    SUM(t.total_value)       AS total_invested
FROM trades t
JOIN accounts a ON t.account_id = a.account_id
GROUP BY a.account_id, t.ticker, t.asset_class;

-- DESCRIBE EXTENDED shows the storage location — it's in the Unity Catalog managed storage
DESCRIBE EXTENDED portfolio_positions_managed;


In [ ]:
%sql
-- External table: Unity Catalog manages only metadata.
-- The data lives at an explicitly specified LOCATION (your ADLS path).
-- When dropped, the data files remain at that location.
--
-- Use case: the risk management system uses Apache Spark directly against the
-- same Parquet files — it doesn't go through Unity Catalog.

-- NOTE: Replace the LOCATION path with a valid external location registered in Unity Catalog.
-- This example uses the Volume path to simulate an external storage location.

CREATE TABLE IF NOT EXISTS portfolio_positions_external
USING DELTA
LOCATION '/Volumes/trainer_demo/demo_06/landing/portfolio_positions/'
AS
SELECT
    a.account_id,
    t.ticker,
    t.asset_class,
    SUM(CASE WHEN t.trade_type = 'Buy' THEN t.quantity
             WHEN t.trade_type = 'Sell' THEN -t.quantity
             ELSE 0 END)   AS net_quantity,
    SUM(t.total_value)       AS total_invested
FROM trades t
JOIN accounts a ON t.account_id = a.account_id
GROUP BY a.account_id, t.ticker, t.asset_class;

-- DESCRIBE EXTENDED shows the storage location points to YOUR specified path
DESCRIBE EXTENDED portfolio_positions_external;


In [ ]:
%sql
-- Compare drop behavior to illustrate the key difference.
-- Show the tables exist first, then drop them.

SHOW TABLES IN trainer_demo.demo_06 LIKE 'portfolio_positions*';


In [ ]:
%sql
-- Drop the managed table: metadata removed, data scheduled for deletion after 8 days.
-- You CAN undo this with UNDROP TABLE within the 8-day window.
DROP TABLE IF EXISTS trainer_demo.demo_06.portfolio_positions_managed;

-- Drop the external table: only metadata removed, data files REMAIN at the LOCATION.
-- There is no UNDROP for external tables.
DROP TABLE IF EXISTS trainer_demo.demo_06.portfolio_positions_external;

-- The files at /Volumes/trainer_demo/demo_06/landing/portfolio_positions/ still exist
-- even after the external table is dropped.
SELECT 'External table dropped — data files still exist at the LOCATION path' AS note;


## Design and implement a clustering strategy

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/design-implement-data-modeling-unity-catalog.design-clustering-strategy.png)

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/design-implement-data-modeling-unity-catalog/media/11-understand-data-skipping-fundamentals.png)

### 🎯 DEMO: Liquid clustering for multi-dimensional trade queries

Compliance analysts at FinServ Demo query trades in many different ways:
- "All EMEA trades for Q1 2023" → filters on `region` AND `trade_date`
- "All JPM equity buys" → filters on `ticker` AND `asset_class` AND `trade_type`
- "All large-value settled trades" → filters on `status` AND `total_value`

Traditional partitioning only helps with **one** dimension. **Liquid clustering** colocates data across multiple filter columns simultaneously, making all three query patterns fast.

> **Trainer tip**: Show that after `OPTIMIZE`, queries with different filter combinations all benefit from data skipping — no partition key needed.


In [ ]:
%sql
-- Create a liquid clustered version of the trades table.
-- CLUSTER BY up to 4 columns that appear most frequently in filters.
-- Unlike PARTITIONED BY, you can change clustering keys later without rewriting data.

USE trainer_demo.demo_06;

CREATE OR REPLACE TABLE trades_clustered
USING DELTA
CLUSTER BY (trade_date, region, asset_class, status)
AS
SELECT * FROM trades;

DESCRIBE EXTENDED trades_clustered;


In [ ]:
%sql
-- Run OPTIMIZE to physically apply liquid clustering to the data files.
-- Delta Lake reorganizes data so rows with similar filter-column values
-- are stored in the same files — enabling effective data skipping.

OPTIMIZE trainer_demo.demo_06.trades_clustered;


In [ ]:
%sql
-- Query 1: EMEA trades in Q1 2023 — benefits from trade_date + region clustering
SELECT COUNT(*) AS emea_q1_trades, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_clustered
WHERE region = 'EMEA'
  AND trade_date BETWEEN '2023-01-01' AND '2023-03-31';


In [ ]:
%sql
-- Query 2: Settled equity trades — benefits from asset_class + status clustering
SELECT ticker, trade_type, SUM(total_value) AS total_value
FROM trainer_demo.demo_06.trades_clustered
WHERE asset_class = 'Equity'
  AND status = 'Settled'
GROUP BY ticker, trade_type
ORDER BY total_value DESC
LIMIT 10;


In [ ]:
%sql
-- Update clustering keys without rewriting the table.
-- Query patterns have changed: analysts now also filter frequently on ticker.
-- With liquid clustering, we can add ticker to the clustering strategy incrementally.

ALTER TABLE trainer_demo.demo_06.trades_clustered
CLUSTER BY (trade_date, region, ticker, status);

-- Next OPTIMIZE run will apply the new clustering going forward.
-- Previously written files are NOT automatically rewritten — use OPTIMIZE FULL for that.
-- OPTIMIZE trainer_demo.demo_06.trades_clustered FULL;   -- uncomment to rewrite all data

SELECT 'Clustering keys updated — next OPTIMIZE will reorganize new data' AS note;


In [ ]:
%sql
-- Enabling deletion vectors improves UPDATE/DELETE performance on the trades table.
-- Instead of rewriting entire Parquet files for each correction, Delta Lake marks
-- modified rows in a separate deletion vector file and merges on read.
-- This is especially valuable for the frequent compliance corrections at FinServ Demo.

ALTER TABLE trainer_demo.demo_06.trades_clustered
SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Simulate a compliance correction — this is now much faster with deletion vectors
UPDATE trainer_demo.demo_06.trades_clustered
SET status = 'Settled'
WHERE status = 'Failed' AND trade_date < '2023-03-01';

DESCRIBE EXTENDED trainer_demo.demo_06.trades_clustered;
